# Entropy and Information

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 03.01–03.11 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/01_entropy_and_information.ipynb)

---

## 🎯 Learning Objective

Define Shannon entropy as expected surprisal, and derive the maximum entropy principle.

---

## 📐 Theory

Suppose someone tells you the outcome of a coin flip. If the coin is fair, you learn something
genuinely new — either answer was equally likely. If the coin always lands heads, being told
"it's heads" tells you nothing you couldn't have guessed. Information theory turns this
intuition — *rare, unpredictable outcomes carry more information than expected, predictable
ones* — into an exact quantity you can compute.

### Surprisal: quantifying how unexpected an outcome is

Let $X$ be a discrete random variable taking values in a finite set $\mathcal{X}$, with
probability mass function $P(x) = \Pr(X=x)$ (recall the PMF from
[`03.02`](../03_Probability_and_Statistics/02_random_variables_distributions.ipynb)). Define the
**self-information**, or **surprisal**, of an outcome $x$ as

$$I(x) = -\log_2 P(x) = \log_2\frac{1}{P(x)}$$

The logarithm isn't an arbitrary choice. Shannon required information to be **additive for
independent events**: learning two unrelated facts should give the sum of what each gives
alone. If $x,y$ are independent, $P(x,y)=P(x)P(y)$, and the only way to turn that product into a
sum while keeping $I$ decreasing in probability is a logarithm:
$I(x,y) = -\log_2(P(x)P(y)) = I(x)+I(y)$. The base just sets the unit: base 2 gives **bits**
(this notebook's convention throughout — "how many yes/no questions to pin down this outcome?"),
base $e$ gives **nats** (common in ML frameworks, since PyTorch's loss functions default to
natural log — flagged again in `05.03`). A certain event ($P(x)=1$) has zero surprisal; an event
with $P(x)=\tfrac12$ carries exactly 1 bit; as $P(x)\to 0$, surprisal grows without bound.

### Shannon entropy: the expected surprisal

**Shannon entropy** is the *average* surprisal of sampling from $X$ — how unpredictable $X$ is
overall, not any single outcome:

$$H(X) = \mathbb{E}_{x\sim P}[I(x)] = -\sum_{x\in\mathcal{X}} P(x)\log_2 P(x)$$

By convention $0\log_2 0 \triangleq 0$ (justified since $p\log_2 p \to 0$ as $p\to 0^+$) — an
outcome that never happens contributes no expected surprisal. Three properties matter before
touching code:

1. **Non-negativity**: $H(X)\ge 0$, with equality iff $X$ is deterministic.
2. **Maximized by uniformity**: for fixed $n=|\mathcal{X}|$, $H(X)$ is maximized when $P$ is
   uniform, giving $H(X)=\log_2 n$ — spreading probability out maximizes average
   unpredictability. Any deviation from uniform strictly lowers entropy.
3. **Concavity**: $H$ is concave in $P$, which is why mixing two distributions never decreases
   entropy on average — used implicitly whenever "smoothing" a distribution (e.g. label
   smoothing) is described as increasing uncertainty.

### The maximum entropy principle

Often you don't know the full distribution $P$ — only a few facts about it (support, mean,
maybe variance). The **maximum entropy principle** (Jaynes, 1957) says: among all distributions
consistent with what you actually know, pick the one with the *highest* entropy. Any other
choice smuggles in extra assumptions your data doesn't support — maximum entropy is the
distribution that is maximally honest about remaining uncertainty.

Formally, to maximize $H(X)$ subject to $\sum_x P(x)=1$ and a linear constraint like
$\sum_x x\,P(x)=\mu$, introduce a Lagrange multiplier $\lambda$ for the mean constraint. The
resulting stationarity condition produces an **exponential-family** solution,
$P(x)\propto e^{\lambda x}$ — the same functional form as the Boltzmann distribution in
statistical mechanics, and (not coincidentally) the same form the softmax function produces.
With no constraints beyond normalization, the max-entropy solution is simply uniform, recovering
property 2 above. We verify this construction numerically below: among all distributions over a
loaded die's six faces with a fixed mean, the max-entropy solution has provably higher entropy
than any other distribution sharing that mean.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Two views of the same idea: surprisal blows up as an outcome becomes rare, while entropy (the
*average* surprisal) is greatest exactly when outcomes are hardest to predict — a fair coin.

In [ ]:
# Surprisal blows up as p->0 (left); entropy of a coin peaks at p=0.5 (right)
p_grid = np.linspace(1e-3, 1, 400)
surprisal = -np.log2(p_grid)

# Entropy of a Bernoulli(p) coin: H(p) = -p log2(p) - (1-p) log2(1-p)
p_coin = np.linspace(1e-6, 1 - 1e-6, 400)
H_coin = -p_coin * np.log2(p_coin) - (1 - p_coin) * np.log2(1 - p_coin)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(p_grid, surprisal, color="#4C72B0", lw=2)
axes[0].scatter([0.5, 0.1, 0.9], [-np.log2(0.5), -np.log2(0.1), -np.log2(0.9)],
                color="#DD8452", zorder=5)
for p_mark in [0.5, 0.1, 0.9]:
    axes[0].annotate(f"p={p_mark}\n{-np.log2(p_mark):.2f} bits", (p_mark, -np.log2(p_mark)),
                      textcoords="offset points", xytext=(10, 8), fontsize=9)
axes[0].set_xlabel("P(x)"); axes[0].set_ylabel("surprisal I(x) = -log2 P(x)  [bits]")
axes[0].set_title("Rarer outcomes carry more surprisal")
axes[0].set_ylim(0, 6)

axes[1].plot(p_coin, H_coin, color="#55A868", lw=2)
axes[1].axvline(0.5, color="gray", ls="--", lw=1)
axes[1].scatter([0.5], [1.0], color="#DD8452", zorder=5)
axes[1].annotate("max entropy: fair coin\nH=1 bit", (0.5, 1.0), textcoords="offset points",
                  xytext=(10, -15), fontsize=9)
axes[1].set_xlabel("P(heads)"); axes[1].set_ylabel("H(coin)  [bits]")
axes[1].set_title("Entropy of a biased coin: uniform = most unpredictable")
plt.tight_layout()
plt.show()

print(f"Entropy at p=0.5 (fair coin):   {H_coin[np.argmin(np.abs(p_coin-0.5))]:.4f} bits (expect 1.0)")
print(f"Entropy at p=0.01 (nearly certain tails): {(-0.01*np.log2(0.01) - 0.99*np.log2(0.99)):.4f} bits")
print("A nearly-certain coin has almost nothing to tell you on average -- low entropy,")
print("even though the rare outcome (heads) itself would be very surprising if it occurred.")

## 🐍 Implementation from Scratch

We implement surprisal and entropy directly from their definitions, cross-check against
`scipy.stats.entropy`, then solve the maximum entropy principle numerically for a die with a
known mean and confirm no other distribution sharing that mean has higher entropy.

In [ ]:
from scipy.stats import entropy as scipy_entropy
from scipy.optimize import brentq

def surprisal_bits(p):
    """I(x) = -log2 P(x). Caller guarantees p > 0 -- an outcome with probability
    zero is never actually observed, so its surprisal is never evaluated."""
    return -np.log2(p)

def entropy_bits(p, eps=1e-12):
    """H(X) = -sum p*log2(p), with the standard convention 0*log2(0) = 0 (implemented
    by clipping only INSIDE the log, since p itself is still the correct weight)."""
    p = np.asarray(p, dtype=float)
    p_safe = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log2(p_safe))

# --- Sanity check against scipy.stats.entropy (base=2 matches our bit convention) ---
test_dists = {
    "fair coin":    np.array([0.5, 0.5]),
    "loaded coin":  np.array([0.9, 0.1]),
    "fair die":     np.full(6, 1 / 6),
}
print("Entropy: our implementation vs scipy.stats.entropy")
for name, p in test_dists.items():
    ours, theirs = entropy_bits(p), scipy_entropy(p, base=2)
    print(f"  {name:12s}: ours={ours:.4f}  scipy={theirs:.4f}  match={np.isclose(ours, theirs)}")
print(f"Uniform bound: H(fair die)={entropy_bits(test_dists['fair die']):.4f} bits vs log2(6)={np.log2(6):.4f} bits")

# --- Maximum entropy principle: find the max-entropy PMF over a die's 6 faces subject only
# to a fixed mean, then confirm no OTHER distribution sharing that mean beats it ---
faces = np.arange(1, 7)
mu_target = 4.5  # a die biased toward high faces, e.g. by physical weighting

def maxent_pmf(lam):
    # Exponential-family form derived from Lagrangian stationarity: P(x) ~ exp(lambda * x)
    w = np.exp(lam * faces)
    return w / w.sum()

def mean_minus_target(lam):
    return np.sum(faces * maxent_pmf(lam)) - mu_target

# Solve for the lambda that hits the target mean exactly (monotonic in lambda, so bisection works)
lam_star = brentq(mean_minus_target, -5, 5)
p_maxent = maxent_pmf(lam_star)

print(f"\nMax-entropy die PMF with mean={mu_target} (solved lambda={lam_star:.4f}):")
print("  P =", np.round(p_maxent, 4))
print(f"  mean check: {np.sum(faces * p_maxent):.4f}  (target {mu_target})")
print(f"  H(max-entropy) = {entropy_bits(p_maxent):.4f} bits")

# Compare against OTHER distributions that ALSO have mean 4.5 but aren't max-entropy --
# they must all have strictly lower entropy, by the theorem. Random Dirichlet draws rarely
# land close to a fixed mean by chance, so draw a large pool and keep only near-matches.
rng = np.random.default_rng(0)
candidates = rng.dirichlet(np.ones(6), size=200_000)
close_enough = np.abs(candidates @ faces - mu_target) < 0.02
competitor_entropies = np.array([entropy_bits(p) for p in candidates[close_enough]])

print(f"\n{close_enough.sum()} random competitor PMFs found with mean within 0.02 of {mu_target}:")
print(f"  H(max-entropy solution)  = {entropy_bits(p_maxent):.4f} bits")
print(f"  H(best competitor found) = {competitor_entropies.max():.4f} bits")
violations = np.sum(competitor_entropies > entropy_bits(p_maxent) + 1e-9)
print(f"  competitors beating the max-entropy solution: {violations} (must be 0, by the theorem)")

## 🤖 Why This Matters for AI

A model's predictive distribution over the next token is a PMF, so entropy directly measures
*how unsure the model is* — high entropy means many tokens are nearly equally plausible, low
entropy means confidence. This is exactly what sampling **temperature** controls at inference:
dividing logits by $T$ before the softmax (`03.02`'s softmax-as-distribution idea) reshapes the
predictive distribution, and entropy measures the effect precisely. Separately, entropy sets a
hard floor on compression: Shannon's source coding theorem says no lossless code can beat
$H(X)$ bits per symbol on average. Tokenizer design (`05.06`) exploits this directly — BPE
merges frequent character pairs into single tokens specifically to lower this floor.

In [ ]:
# Sampling temperature reshapes a language model's next-token distribution -- entropy
# measures exactly how much more (or less) uncertain that makes the model's output.
rng = np.random.default_rng(0)
vocab = ["the", "a", "cat", "dog", "runs", "sleeps", "quickly", "outside", "yesterday", "!"]
logits = rng.normal(scale=2.0, size=len(vocab))  # a toy next-token logit vector

def softmax(z):
    z = z - z.max()               # subtract max for numerical stability (Module 06 covers this trick in depth)
    e = np.exp(z)
    return e / e.sum()

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
print(f"{'T':>5} | {'entropy (bits)':>15} | {'top token':>10} | {'top prob':>9}")
print("-" * 50)
entropies = []
for T in temperatures:
    p = softmax(logits / T)
    H = entropy_bits(p)
    entropies.append(H)
    top_i = np.argmax(p)
    print(f"{T:5.1f} | {H:15.3f} | {vocab[top_i]:>10} | {p[top_i]:9.3f}")

max_possible = np.log2(len(vocab))
print(f"\nMaximum possible entropy for a {len(vocab)}-token vocabulary: log2({len(vocab)}) = {max_possible:.3f} bits")
print(f"At T=5.0 the distribution is nearly uniform, so entropy ({entropies[-1]:.3f}) approaches that ceiling.")
print(f"At T=0.1 the model is almost certain -- entropy collapses to {entropies[0]:.3f} bits, close to 0.")
print("This is exactly why low temperature makes LLM output feel deterministic/repetitive,")
print("and high temperature makes it feel diverse (and eventually incoherent).")

# --- Compression connection: entropy sets the average-bits-per-symbol floor (source coding theorem) ---
print("\n--- Entropy as a compression bound ---")
text = ("the quick brown fox jumps over the lazy dog while the cat sleeps near the "
        "warm fireplace and the mice scurry quietly across the wooden floor at night")
chars, counts = np.unique(list(text), return_counts=True)
H_text = entropy_bits(counts / counts.sum())
naive_fixed_width = np.log2(len(chars))  # bits/char if every symbol got an equal-length code

print(f"Text: {len(text)} characters, {len(chars)} distinct symbols.")
print(f"Empirical entropy: {H_text:.3f} bits/char  |  naive fixed-width: {naive_fixed_width:.3f} bits/char  |  ASCII: 8.0 bits/char")
print(f"Shannon's source coding theorem: no LOSSLESS code beats {H_text:.3f} bits/char on average here --")
print("entropy is a hard floor, not just a target. A good tokenizer (`05.06`) shapes the token")
print("distribution to push this floor as low as possible.")

## 🏋️ Exercises

### Warm-up
1. By hand, compute the surprisal $I(x) = -\log_2 P(x)$ for an outcome with $P(x) = 1/8$, and
   the entropy of a fair 8-sided die. Verify both with `surprisal_bits`/`entropy_bits`.

### Practice
2. Compute the entropy of a biased coin with $P(\text{heads})=0.05$. Then find, by trial and
   error or with `scipy.optimize.brentq` (as in `maxent_pmf`'s use above), the value of $p$ that
   makes a *three*-outcome categorical distribution $(p, p, 1-2p)$ have maximum entropy, and
   confirm it matches the uniform-distribution prediction from this notebook's Theory section.

### Challenge
3. In the AI section, `vocab` has 10 tokens. Add 20 more low-probability "filler" tokens to the
   vocabulary (extend `logits` with small random values) without changing the original 10
   logits, and recompute entropy at $T=1.0$. Explain in a sentence or two why entropy increases
   even though the model's *actual* top predictions barely changed — and connect this to why
   vocabulary size is itself a design choice that affects a tokenizer's baseline uncertainty
   (previewed further in `05.06`).

---

## 📚 Further Reading
- Shannon, ["A Mathematical Theory of Communication"](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf) (1948) — the original paper defining entropy
- Jaynes, ["Information Theory and Statistical Mechanics"](https://bayes.wustl.edu/etj/articles/theory.1.pdf) (1957) — the maximum entropy principle
- Cover & Thomas, *Elements of Information Theory*, Ch. 2 (Entropy, Relative Entropy, and Mutual Information)

---

*Next notebook: [KL Divergence](02_kl_divergence.ipynb)*